# dARK Minter API - Test Notebook

This notebook tests the `dark-core-minter-api` endpoints using FastAPI's TestClient.

It covers:
1.  **Setup**: Initializing the Orchestrator and TestClient.
2.  **Health Check**: Verifying API availability.
3.  **Authority**: Setting up a test authority and querying it via API.
4.  **Minting**: Creating (Minting) new ARKs via the API.

**Note**: This API does NOT support resolving or looking up ARKs.


## 1. Setup Environment

In [ ]:
import os
import sys
import logging
from dotenv import load_dotenv

# Add app to path
sys.path.append(os.path.abspath('..'))

# Logging
logging.basicConfig(level=logging.INFO)

# Load unified configuration
if os.path.exists('../../.env'):
    load_dotenv('../../.env')
    print("Loaded ../../.env")
elif os.path.exists('../.env'):
    load_dotenv('../.env')
    print("Loaded ../.env")
else:
    print('⚠️  Warning: No .env found')

## 2. Initialize App & TestClient

In [ ]:
from fastapi.testclient import TestClient
from app.main import app
from app.dependencies import get_orchestrator

# Create TestClient
# This triggers the lifespan events (startup) -> initializes orchestrator
client = TestClient(app)

print("✅ TestClient initialized")

## 3. Health Check

In [ ]:
response = client.get("/health")
print(f"Status Code: {response.status_code}")
print(f"Response: {response.json()}")
assert response.status_code == 200

## 4. Setup Test Authority (Direct Orchestrator Access)

We need a registered authority to test minting. We'll use the orchestrator directly to set one up, similar to the lib test.

In [ ]:
import time

# Get the orchestrator instance used by the app
orchestrator = get_orchestrator()

# Setup Authority
UUID = f"minter-test-{int(time.time())}"
NAAN = "12345"
NAANS = [NAAN]

print(f"Setting up authority: {UUID}")
authority = orchestrator.setup_authority(UUID, NAANS)
print(f"✅ Authority Setup: {authority.wallet_address}")

## 5. Test Authority Endpoint

Verify the API can retrieve the authority we just created.

In [ ]:
response = client.get(f"/api/v1/authority/{UUID}")

print(f"Status: {response.status_code}")
print(f"Body: {response.json()}")

assert response.status_code == 200
data = response.json()
assert data['uuid'] == UUID
assert NAAN in data['naans']

## 6. Test Batch Mint Endpoint

Use the `/api/v1/mint/batch` endpoint to mint a new ARK.

In [ ]:
NAME = f"test-doc-{int(time.time())}"
URL = "https://example.org/doc/1"
CID = "bafybeigdyrzt5sfp7udbbkc5dla2yv5ifyrkkwdxgper"

payload = {
    "items": [
        {
            "authority_id": UUID,
            "naan": NAAN,
            "name": NAME,
            "url": URL,
            "cid": CID
        }
    ]
}

print(f"Minting ark:/{NAAN}/{NAME} ...")
response = client.post("/api/v1/mint/batch", json=payload)

print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")

assert response.status_code == 200
result = response.json()
assert result['status'] == 'ok'
assert result['results'][0]['status'] == 'success'

## 7. Verify NO Resolve Endpoint

Ensure that trying to reach resolve endpoints returns 404.

In [ ]:
response = client.get(f"/api/v1/resolve/{NAAN}/{NAME}")
print(f"Resolve Endpoint Status: {response.status_code}")
assert response.status_code == 404

response = client.post("/api/v1/lookup/url", json={"url": URL})
print(f"Lookup Endpoint Status: {response.status_code}")
assert response.status_code == 404